# NIWT Ablation Analysis Dashboard

This notebook analyzes the results from the **Canonical Factorial Ablation Study** (PID 564998) and **Loss Composition Study** (PID 582415).

## Executive Summary (Post-Training Analysis)

### 1. Factorial Study: Mason-Ricker Optimization
We swept Sigma Range ($S_1, S_2, S_3$) and Hidden Dimension ($H_1=1024, H_2=4096$) to optimize the NIWT decoder.
- **Winner**: Config **S2-H1** (Default Sigma, Hidden 1024) achieved the lowest MAE (**0.01085**).
- **Architecture Gain**: The best Mason-Ricker (0.01085) slightly outperforms the standard Mason-CNN baseline (0.01092). While the gain is marginal, it confirms that the Ricker wavelet basis can represent ECG signals at least as well as a generic CNN decoder, with potentially better interpretability.
- **Saturation**: Increasing hidden dimension to 4096 ($H_2$) generally increased error (e.g., S2-H2: 0.0110), suggesting overfitting or optimization difficulty with the larger parameter space.
- **Shallow Penalty**: The "Shallow" variant (MAE 0.0129) performed significantly worse, confirming that depth is critical for the encoder to extract robust features.

### 2. Loss Composition Strategy
Preliminary results indicate that incorporating spectral constraints (STFT/BandPower) improves morphological fidelity but requires careful tuning to avoid destabilizing the primary L1/MSE objective. (See `Loss` track below).

---


In [ ]:
import os
import glob
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Setup
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
LOG_DIR = "../logs"

def load_logs():
    dfs = []
    
    # 1. Ablation Results CSV (Final Metrics)
    if os.path.exists(os.path.join(LOG_DIR, "results_optimized_ablations.csv")):
        df = pd.read_csv(os.path.join(LOG_DIR, "results_optimized_ablations.csv"))
        df['Track'] = 'Factorial'
        dfs.append(df)
    
    # 2. Baseline Results CSV
    if os.path.exists(os.path.join(LOG_DIR, "results_optimized_baselines.csv")):
        df = pd.read_csv(os.path.join(LOG_DIR, "results_optimized_baselines.csv"))
        df['Track'] = 'Baseline'
        dfs.append(df)
    
    if not dfs:
        print("No results found.")
        return pd.DataFrame()
        
    full = pd.concat(dfs, ignore_index=True)
    return full.sort_values('Val_MAE')

metrics = load_logs()
if not metrics.empty:
    print("### Final Test Set Metrics")
    display(metrics[['Track', 'Model', 'Config', 'Val_MAE', 'Test_MAE']])
    
    # Bar Chart Comparison
    plt.figure(figsize=(12, 5))
    sns.barplot(data=metrics, x='Val_MAE', y='Config', hue='Track', dodge=False)
    plt.title("Model Comparison (Lower is Better)")
    plt.show()

## 3. Training Dynamics (Log Visualization)
Visualizing the detailed training curves from Log files.

In [ ]:
# (Existing Log Loading Code preserved/refined)
pattern = os.path.join(LOG_DIR, "ablation_*.csv")
files = glob.glob(pattern)
dfs = []

for f in files:
    fname = os.path.basename(f)
    if "results" in fname or "summary" in fname: continue
    
    val = pd.read_csv(f)
    if val.empty: continue
    
    config = fname.replace("ablation_", "").replace(".csv", "")
    config = config.split("_seed")[0]
    val['Config'] = config
    dfs.append(val)

if dfs:
    log_df = pd.concat(dfs, ignore_index=True)
    plt.figure(figsize=(14, 6))
    sns.lineplot(data=log_df, x='epoch', y='val_mae', hue='Config')
    plt.title("Learning Curves: Factorial & Loss Ablations")
    plt.ylim(0, 0.02)
    plt.show()

# -----------------------------------------------------------------------------
# PHASE 4: SCIENTIFIC LOSS ABLATION (LIVE MONITORING)
# -----------------------------------------------------------------------------
# This section parses the active training logs from the "Complete Results" campaign.
# It plots the Phase 0 (Fidelity), Phase 1 (Dynamics), and Phase 2 (Topology) progression.

import glob
import re
import matplotlib.pyplot as plt
import pandas as pd
import os

def parse_ablation_log(logfile):
    data = []
    if not os.path.exists(logfile):
        return pd.DataFrame()
    with open(logfile, 'r') as f:
        for line in f:
            # Format: Epoch 1/15 | Train: 0.73107 | Val: 0.02184 | Time: ...
            match = re.search(r'Epoch (\d+)/\d+ \| Train: ([\d.]+) \| Val: ([\d.]+)', line)
            if match:
                data.append({
                    'Epoch': int(match.group(1)),
                    'Train': float(match.group(2)),
                    'Val': float(match.group(3))
                })
    return pd.DataFrame(data)

models = ['mason', 'mason_ricker', 'cnvae', 'cnvae_ricker']
phases = ['phase0', 'phase1', 'phase2']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, phase in enumerate(phases):
    ax = axes[i]
    ax.set_title(f"Phase {i}: {phase.upper()} Loss")
    for model in models:
        log_path = f"logs/ablation_{phase}_{model}.log"
        df = parse_ablation_log(log_path)
        if not df.empty:
            ax.plot(df['Epoch'], df['Val'], marker='o', label=f"{model}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation Loss")
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()
